# Práctica — Ejercicio 5: Imputación de precios de alquiler en Buenos Aires

Usando el dataset `listings_ba.csv`, imputar los **100 precios de alquiler faltantes** con tres métodos distintos.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
df = pd.read_csv("../Datasets/listings_ba.csv")
print(f"Shape: {df.shape}")
print(f"Precios faltantes: {df['price'].isna().sum()}")
df[['id', 'neighbourhood', 'room_type', 'latitude', 'longitude', 'price']].head(3)

## Ítem 1 — Imputación por media y moda

> **La media y moda de los datos no faltantes. En cada uno de los casos indicar la cantidad de datos que se usaron para la imputación.**

In [ ]:
df_conocidos = df[df['price'].notna()]
media = df_conocidos['price'].mean()
moda = df_conocidos['price'].mode()[0]

print(f"Datos no faltantes usados para calcular media/moda: {len(df_conocidos)}")
print(f"Media:  {media:.2f}")
print(f"Moda:   {moda:.2f}")

# Imputación con media
df_media = df.copy()
df_media['price'] = df_media['price'].fillna(media)

# Imputación con moda
df_moda = df.copy()
df_moda['price'] = df_moda['price'].fillna(moda)

print(f"\nNaN después de imputar con media: {df_media['price'].isna().sum()}")
print(f"NaN después de imputar con moda:  {df_moda['price'].isna().sum()}")

## Ítem 2 — Imputación por barrio y tipo de habitación

> **Una medida de resumen de su elección por barrio y tipo de habitación. Indicar la cantidad de datos que se usaron para la imputación.**

In [ ]:
# Mediana por neighbourhood + room_type (más robusta que la media ante outliers de precio)
medianas = (df[df['price'].notna()]
            .groupby(['neighbourhood', 'room_type'])['price']
            .median())

df_grupo = df.copy()
mask_nan = df_grupo['price'].isna()
faltantes = df_grupo[mask_nan][['neighbourhood', 'room_type']]

imputados = 0
sin_grupo = 0
for idx, row in faltantes.iterrows():
    clave = (row['neighbourhood'], row['room_type'])
    if clave in medianas.index:
        df_grupo.at[idx, 'price'] = medianas[clave]
        imputados += 1
    else:
        sin_grupo += 1

print(f"Grupos (neighbourhood × room_type) usados para calcular medianas: {len(medianas)}")
print(f"Registros imputados con mediana de grupo: {imputados}")
print(f"Registros sin grupo disponible (NaN restantes): {sin_grupo}")
print(f"NaN restantes en df_grupo: {df_grupo['price'].isna().sum()}")

## Ítem 3 — Imputación por los 10 vecinos geográficos más cercanos

> **Los 10 puntos más cercanos geográficamente a cada dato faltante usando las coordenadas. Indicar la cantidad de datos que se usaron para la imputación.**

In [ ]:
# KNN manual usando distancia euclidiana en coordenadas (lat, lon)
# Nota: para distancias pequeñas la distancia euclidiana es una buena aproximación

K = 10

coords_conocidos = df_conocidos[['latitude', 'longitude']].values
precios_conocidos = df_conocidos['price'].values

df_knn = df.copy()
mask_nan = df_knn['price'].isna()
coords_faltantes = df_knn[mask_nan][['latitude', 'longitude']].values

precios_imputados = []
for coord in coords_faltantes:
    distancias = np.sqrt(np.sum((coords_conocidos - coord) ** 2, axis=1))
    idx_cercanos = np.argsort(distancias)[:K]
    precio_medio = np.mean(precios_conocidos[idx_cercanos])
    precios_imputados.append(precio_medio)

df_knn.loc[mask_nan, 'price'] = precios_imputados

print(f"Datos no faltantes usados como referencia para KNN: {len(df_conocidos)}")
print(f"K (vecinos más cercanos): {K}")
print(f"Registros imputados: {mask_nan.sum()}")
print(f"NaN restantes: {df_knn['price'].isna().sum()}")

In [ ]:
# Comparación de las distribuciones de precio según método
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, (titulo, datos) in zip(axes, [
    ('Media global', df_media),
    ('Mediana por grupo', df_grupo),
    ('KNN geográfico (k=10)', df_knn),
]):
    ax.hist(datos['price'].clip(upper=datos['price'].quantile(0.99)),
            bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(titulo)
    ax.set_xlabel('Precio')
    ax.set_ylabel('Frecuencia')

plt.suptitle('Distribución de precios según método de imputación', y=1.02)
plt.tight_layout()
plt.show()

## Conclusiones

| Método | Datos usados | Característica |
|--------|-------------|----------------|
| Media global | 19.985 obs. | Simple, ignora variación geográfica y tipo de habitación |
| Mediana por barrio + tipo | Varía por grupo | Más informativa, captura variación por categoría |
| KNN geográfico (k=10) | 10 vecinos más cercanos | Aprovecha la ubicación espacial; captura heterogeneidad geográfica |

El **KNN geográfico** es el método más informativo para datos de alquiler, ya que los precios tienen fuerte componente espacial.